<a href="https://colab.research.google.com/github/kumarmohit0911/AAA/blob/main/GAN_MNIST_Fashion_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader
import torchvision.datasets as datasets
import torchvision.transforms as transform

In [2]:
class Discriminator(nn.Module):
  def __init__(self,img_dim):
    super().__init__()
    self.disc = nn.Sequential(
        nn.Linear(img_dim,128),
        nn.LeakyReLU(0.1),
        nn.Linear(128,1),
        nn.Sigmoid()
    )
  def forward(self,x):
    return self.disc(x)

In [3]:
class Generator(nn.Module):
  def __init__(self,z_dim,img_dim):
    super().__init__()
    self.disc = nn.Sequential(
        nn.Linear(z_dim,256),
        nn.LeakyReLU(0.1),
        nn.Linear(256,img_dim),
        nn.Tanh()
    )
  def forward(self,x):
    return self.disc(x)

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [5]:
lr = 3e-4
z_dim = 64
img_dim = 28*28*1
batch_size = 32
epochs = 50

In [6]:
# initialising
disc = Discriminator(img_dim).to(device)
gen = Generator(z_dim,img_dim).to(device)
fixed_noise = torch.randn((batch_size,z_dim)).to(device)
transforms = transform.Compose(
    [transform.ToTensor(),
     transform.Normalize((0.5,),(0.5,))]
)

In [7]:
dataset = datasets.MNIST(root="dataset/",transform=transforms,download=True)
loader = DataLoader(dataset,batch_size = batch_size,shuffle=True)

100%|██████████| 9.91M/9.91M [00:00<00:00, 50.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.71MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 15.0MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.35MB/s]


In [8]:
# creating optimizer
opt_disc = optim.Adam(disc.parameters(),lr = lr)
optim_gen = optim.Adam(gen.parameters(),lr = lr)
criterion = nn.BCELoss()

In [ ]:
for epoch in range(epochs):
  for batch_idx,(real,_) in enumerate(loader):
    real= real.view(-1,784).to(device)
    batch_size=real.shape[0]
    # training Discriminator
    noise = torch.randn(batch_size,z_dim).to(device)
    fake = gen(noise)
    disc_real = disc(real).view(-1)
    lossD_real = criterion(disc_real,torch.zeros_like(disc_real))
    disc_fake = disc(fake.detach()).view(-1)
    lossD_fake = criterion(disc_fake,torch.zeros_like(disc_fake))
    lossD = (lossD_real+lossD_fake)/2
    disc.zero_grad()
    lossD.backward()
    opt_disc.step()
    # training generator
    output = disc(fake).view(-1)
    lossG=criterion(output,torch.ones_like(output))
    gen.zero_grad()
    lossG.backward()
    optim_gen.step()